## Built-In Evaluators - Measuring Agent Performance with Strands Evals

This tutorial introduces the complete toolkit of built-in evaluators provided by Strands Evals. You'll learn how to measure different aspects of agent performance using standardized evaluation metrics, from response quality to tool selection accuracy.

### What You'll Learn
- Understand the purpose of each built-in evaluator
- Apply OutputEvaluator with domain-specific rubrics
- Use trace-based evaluators (HelpfulnessEvaluator, GoalSuccessRateEvaluator, ToolSelectionAccuracyEvaluator, ToolParameterAccuracyEvaluator)
- Analyze agent reasoning with TrajectoryEvaluator
- Compare evaluation results across different metrics

### Tutorial Details

| Information         | Details                                                                       |
|:--------------------|:------------------------------------------------------------------------------|
| Tutorial type       | Beginner - Introduction to built-in evaluation metrics                        |
| Tutorial components | Recipe Bot agent, 6 built-in evaluators, results visualization               |
| Tutorial vertical   | Agent Evaluation                                                              |
| Example complexity  | Easy                                                                          |
| SDK used            | Strands Agents, Strands Evals                                                 |

### Understanding Built-In Evaluators

Evaluating agent performance is complex because agents operate across multiple dimensions—they generate responses, complete tasks, and use tools to achieve goals. A single metric can't capture all aspects of agent behavior, which is why Strands Evals provides six specialized built-in evaluators.

| Evaluator | Type | Use When | Measures | Requirements |
|:----------|:-----|:---------|:---------|:-------------|
| **OutputEvaluator** | Output-based | Verify correct, complete answers | Correctness, completeness, relevance via custom rubrics | None |
| **HelpfulnessEvaluator** | Trace-based | Ensure agent is genuinely useful | Practical value, clarity, actionability (7-point scale) | OpenTelemetry and Session mapping |
| **GoalSuccessRateEvaluator** | Trace-based | Track task completion rates | Binary success/failure against defined goals | OpenTelemetry and Session mapping |
| **ToolSelectionAccuracyEvaluator** | Trace-based | Verify proper tool selection | Whether agent chose the right tools | OpenTelemetry and Session mapping |
| **ToolParameterAccuracyEvaluator** | Trace-based | Validate tool parameter usage | Correctness of tool parameter values | OpenTelemetry and Session mapping |
| **TrajectoryEvaluator** | Trajectory-based | Understand agent reasoning | Quality of reasoning steps and action sequences | Trajectory extractor |

#### Important API Note

**ONE Evaluator Per Dataset**: Each Dataset accepts exactly ONE evaluator object. To demonstrate multiple evaluators, we run separate evaluation rounds.

In [1]:
import os

### Environment Setup

Configure AWS region and model settings for this tutorial.

In [2]:
import boto3

# AWS Configuration
session = boto3.Session()
AWS_REGION = session.region_name or 'us-east-1'
DEFAULT_MODEL = 'us.anthropic.claude-3-7-sonnet-20250219-v1:0'

### Setup and Imports

Import all necessary libraries for agent creation and evaluation.

In [3]:
# Standard imports
import json
from typing import List, Dict

# Strands imports
from strands import Agent, tool

# Strands Evals imports
from strands_evals import Dataset, Case
from strands_evals.evaluators import (
    OutputEvaluator,
    HelpfulnessEvaluator,
    GoalSuccessRateEvaluator,
    ToolSelectionAccuracyEvaluator,
    ToolParameterAccuracyEvaluator,
    TrajectoryEvaluator
)
from strands_evals.extractors import tools_use_extractor

# Display utilities
from IPython.display import Markdown, display
import pandas as pd

### Recipe Bot Agent

We'll use a Recipe Bot agent to demonstrate built-in evaluators. This agent helps users find recipes and answers cooking questions using web search.

In [4]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException
import time

# Define a websearch tool
@tool
def websearch(
    keywords: str, region: str = "us-en", max_results: int | None = None
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        time.sleep(15)
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"


# System prompt for Recipe Bot
RECIPE_BOT_SYSTEM_PROMPT = """You are RecipeBot, a helpful cooking assistant.
Help users find recipes based on ingredients and answer cooking questions.
Use the websearch tool to find recipes when users mention ingredients or to look up cooking information."""

### Test the Agent

Before evaluating, let's verify the agent works correctly with a simple test query.

In [5]:
# Create a test agent instance
test_agent = Agent(
    system_prompt=RECIPE_BOT_SYSTEM_PROMPT,
    tools=[websearch],
    model=DEFAULT_MODEL
)

# Test with a simple query
test_query = "What can I make with chicken and tomatoes?"
test_response = test_agent(test_query)

I'd be happy to help you find recipes that use chicken and tomatoes! Let me search for some options for you.
Tool #1: websearch
Based on my search, I've found some great recipes you can make with chicken and tomatoes. Here are several delicious options:

1. **Easy Chicken in Tomato Sauce** - A 30-minute one-pan meal where juicy seared chicken breasts are covered with tomato sauce. This is fast, fresh, and healthy!

2. **Tomato Basil Chicken Breasts** - A piccata-ish chicken dish served with a flavorful sauce made from butter, shallots, tomatoes, and basil.

3. **Baked Chicken with Tomatoes and Herbs** - An easy dish where chicken breasts are baked with fresh tomatoes and herbs for a simple yet flavorful meal.

4. **Grilled Chicken Thighs with Tomatoes** - In this elegant summer meal, chicken thighs are marinated in an herbes de Provence vinaigrette, then grilled alongside tomatoes.

5. **Chicken with Sun-Dried Tomatoes** - A creamy one-skillet dish that's rich and flavorful (and there'

### Create Test Cases

We'll create test cases with domain-specific expectations for Recipe Bot evaluation.

In [6]:
# Create test cases for evaluation
test_cases = [
    Case(
        name="Recipe Search - Simple Ingredients",
        input="I have chicken and broccoli. What can I cook?",
        expected_output="A helpful response with recipe suggestions that include chicken and broccoli, with cooking instructions or search results.",
        metadata={
            "goal": "Find recipes using specified ingredients",
            "expected_tools": ["websearch"],
            "expected_tool_params": {
                "websearch": {
                    "keywords": ["chicken", "broccoli", "recipe"]
                }
            }
        }
    ),
    Case(
        name="Cooking Technique Question",
        input="How do I properly sear a steak?",
        expected_output="Clear instructions on steak searing technique, including temperature, timing, and tips for achieving a good sear.",
        metadata={
            "goal": "Learn proper steak searing technique",
            "expected_tools": ["websearch"],
            "expected_tool_params": {
                "websearch": {
                    "keywords": ["sear", "steak"]
                }
            }
        }
    ),
    Case(
        name="Dietary Restriction Recipe",
        input="Can you find me a vegan pasta recipe?",
        expected_output="One or more vegan pasta recipes with ingredients and preparation steps.",
        metadata={
            "goal": "Find vegan pasta recipes",
            "expected_tools": ["websearch"],
            "expected_tool_params": {
                "websearch": {
                    "keywords": ["vegan", "pasta", "recipe"]
                }
            }
        }
    )
]

### OpenTelemetry Setup

Trace-based evaluators (HelpfulnessEvaluator, GoalSuccessRateEvaluator, ToolSelectionAccuracyEvaluator, ToolParameterAccuracyEvaluator) require OpenTelemetry setup to capture agent execution traces.

In [7]:
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Setup telemetry - CORRECT WAY per README
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()

### Evaluation Round 1: OutputEvaluator

OutputEvaluator assesses response quality using domain-specific rubrics. For Recipe Bot, we check for ingredient lists, cooking instructions, and timing information.

In [8]:
# Create OutputEvaluator with domain-specific rubric
output_evaluator = OutputEvaluator(
    rubric="""Recipe responses should include:
    1. Clear ingredient list with quantities (0-0.3 points)
    2. Step-by-step cooking instructions (0-0.4 points)
    3. Cooking time/temperature if applicable (0-0.3 points)
    Score proportionally based on completeness."""
)

# Create dataset with OutputEvaluator
output_dataset = Dataset[str, str](cases=test_cases, evaluator=output_evaluator)

# Simple task function (no OTEL needed for OutputEvaluator)
def simple_task(case: Case) -> str:
    agent = Agent(
        system_prompt=RECIPE_BOT_SYSTEM_PROMPT,
        tools=[websearch],
        model=DEFAULT_MODEL
    )
    return str(agent(case.input))

# Run evaluation
output_report = output_dataset.run_evaluations(simple_task)

I'd be happy to help you find some recipes using chicken and broccoli! Let me search for some options for you.
Tool #1: websearch
Based on my search, I found several delicious recipes you can make with chicken and broccoli! Here are some popular options:

## Chicken and Broccoli Stir-Fry
This is a quick 15-minute dinner option with a savory sauce. The recipe typically includes:
- Chicken breast pieces
- Broccoli florets
- Garlic
- Soy sauce
- Sesame oil
- Sherry or rice wine

## One-Pan Baked Lemon Parmesan Chicken and Broccoli
A simple sheet pan dinner ready in under 20 minutes:
- Chicken pieces seasoned with lemon
- Roasted broccoli florets
- Parmesan cheese
- Ready in one pan for easy cleanup

## Chicken and Broccoli Casserole
A comforting option that's family-friendly:
- Chicken pieces
- Broccoli florets
- Rice
- Creamy sauce (often made with cream soup)
- Cheese topping
- Baked until bubbly

## Chicken and Broccoli Pasta
Several variations include:
- Chicken and Broccoli Alfredo w

In [9]:
# Display OutputEvaluator results
output_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.37           Pass Rate: 0.3333333333333333                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 0.20  │ ❌        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ Cooking Technique Question         │ 0.70  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ Dietary Restriction Recipe         │ 0.20  │ ❌        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.37           Pass Rate: 0.3333333333333333                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 0.20  │ ❌        │ The output provides        │ I have chicken and        │
│       │ Ingredients                │       │           │ multiple recipe            │ broccoli. What can I      │
│       │                            │       │           │ suggestions but lacks the  │ cook?                     │
│       │                            │       │           │ required components: no    │                           │
│       │                            │       │           │ specific quantities for    │                           │
│       │                            │       │           │ ingredients (only lists    │                           │
│       │                            │       │           │ basic ingredients), no     │                           │
│       │                            │       │           │ step-by-step cooking       │                           │
│       │                            │       │           │ instructions (only         │                           │
│       │                            │       │           │ describes what recipes     │                           │
│       │                            │       │           │ typically include), and    │                           │
│       │                            │       │           │ limited cooking            │                           │
│       │                            │       │           │ time/temperature details   │                           │
│       │                            │       │           │ (mentions "15 minutes" and │                           │
│       │                            │       │           │ "under 20 minutes" but no  │                           │
│       │                            │       │           │ temperatures or detailed   │                           │
│       │                            │       │           │ timing). While helpful as  │                           │
│       │                            │       │           │ inspiration, it doesn't    │                           │
│       │                            │       │           │ meet the rubric's          │                           │
│       │                            │       │           │ requirements for complete  │                           │
│       │                            │       │           │ recipe information.        │                           │
├───────┼────────────────────────────┼───────┼───────────┼────────────────────────────┼───────────────────────────┤
│ ▼ 1   │ Cooking Technique Question │ 0.70  │ ✅        │ The output provides        │ How do I properly sear a  │
│       │                            │       │           │ comprehensive step-by-step │ steak?                    │
│       │                            │       │           │ cooking instructions       │                           │
│       │                            │       │           │ (0.4/0.4) and includes     │                           │
│       │                            │       │           │ detailed cooking           │                           │
│       │                            │       │           │ temperatures and timing    │                           │
│       │                            │       │           │ (0.3/0.3). However, it     │                           │
│       │                            │       │           │ lacks a clear ingredient   │                           │
│       │                            │       │           │

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Evaluation Round 2: HelpfulnessEvaluator

HelpfulnessEvaluator measures how useful the agent's response is to users on a 7-point scale. This evaluator requires OpenTelemetry Session data.

In [18]:
# Create HelpfulnessEvaluator
helpfulness_evaluator = HelpfulnessEvaluator()

# Create dataset
helpfulness_dataset = Dataset[str, str](cases=test_cases, evaluator=helpfulness_evaluator)

# Task function with OTEL support
import uuid

def trace_task(case: Case) -> dict:
    telemetry.in_memory_exporter.clear()
    session_id = str(uuid.uuid4())  # Generate unique session ID
    agent = Agent(
        system_prompt=RECIPE_BOT_SYSTEM_PROMPT,
        tools=[websearch],
        model=DEFAULT_MODEL,
        trace_attributes={"session.id": session_id},
        callback_handler=None
    )
    response = agent(case.input)
    
    # Force flush all spans to ensure they're captured
    telemetry.tracer_provider.force_flush()

    # Map spans to Session
    finished_spans = telemetry.in_memory_exporter.get_finished_spans()
    mapper = StrandsInMemorySessionMapper()
    session = mapper.map_to_session(finished_spans, session_id=session_id)

    return {"output": str(response), "trajectory": session}

# Run evaluation
helpfulness_report = helpfulness_dataset.run_evaluations(trace_task)

In [19]:
helpfulness_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.83           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 0.83  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ Cooking Technique Question         │ 0.83  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ Dietary Restriction Recipe         │ 0.83  │ ✅        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 0.83           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 0.83  │ ✅        │ The user asked what they   │ I have chicken and        │
│       │ Ingredients                │       │           │ can cook with chicken and  │ broccoli. What can I      │
│       │                            │       │           │ broccoli, which is a clear │ cook?                     │
│       │                            │       │           │ goal of getting cooking    │                           │
│       │                            │       │           │ suggestions. The assistant │                           │
│       │                            │       │           │ provided an excellent      │                           │
│       │                            │       │           │ response by:               │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ 1. Directly addressing the │                           │
│       │                            │       │           │ user's question with       │                           │
│       │                            │       │           │ multiple recipe options    │                           │
│       │                            │       │           │ 2. Providing 4 different   │                           │
│       │                            │       │           │ cooking approaches         │                           │
│       │                            │       │           │ (stir-fry, casserole,      │                           │
│       │                            │       │           │ baked, pasta) with clear   │                           │
│       │                            │       │           │ names and brief            │                           │
│       │                            │       │           │ descriptions               │                           │
│       │                            │       │           │ 3. Listing the additional  │                           │
│       │                            │       │           │ ingredients needed for     │                           │
│       │                            │       │           │ each recipe option         │                           │
│       │                            │       │           │ 4. Giving time estimates   │                           │
│       │                            │       │           │ where relevant (15         │                           │
│       │                            │       │           │ minutes, 20 minutes)       │                           │
│       │                            │       │           │ 5. Offering recipes for    │                           │
│       │                            │       │           │ different cooking styles   │                           │
│       │                            │       │           │ and preferences            │                           │
│       │                            │       │           │ 6. Asking a follow-up      │                           │
│       │                            │       │           │ question to potentially    │                           │
│       │                            │       │           │ provide even more targeted │                           │
│       │                            │       │           │ suggestions based on       │                           │
│       │                            │       │           

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Evaluation Round 3: GoalSuccessRateEvaluator

GoalSuccessRateEvaluator determines if the agent successfully completed the user's stated goal (binary success/failure).

In [20]:
# Create GoalSuccessRateEvaluator
goal_evaluator = GoalSuccessRateEvaluator()

# Create dataset
goal_dataset = Dataset[str, str](cases=test_cases, evaluator=goal_evaluator)

# Run evaluation (reuse trace_task function)
goal_report = goal_dataset.run_evaluations(trace_task)


In [21]:
# Display GoalSuccessRateEvaluator results
goal_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ Cooking Technique Question         │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ Dietary Restriction Recipe         │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 1.00  │ ✅        │ The user asked "I have     │ I have chicken and        │
│       │ Ingredients                │       │           │ chicken and broccoli. What │ broccoli. What can I      │
│       │                            │       │           │ can I cook?" Their goal    │ cook?                     │
│       │                            │       │           │ was to get recipe ideas    │                           │
│       │                            │       │           │ for dishes they can make   │                           │
│       │                            │       │           │ with these two             │                           │
│       │                            │       │           │ ingredients.               │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ To analyze this:           │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ 1. The AI assistant should │                           │
│       │                            │       │           │ use a web search tool to   │                           │
│       │                            │       │           │ find relevant chicken and  │                           │
│       │                            │       │           │ broccoli recipes, then     │                           │
│       │                            │       │           │ provide the user with      │                           │
│       │                            │       │           │ multiple recipe options    │                           │
│       │                            │       │           │ based on the search        │                           │
│       │                            │       │           │ results.                   │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ 2. Looking at the          │                           │
│       │                            │       │           │ conversation record:       │                           │
│       │                            │       │           │ - The assistant correctly  │                           │
│       │                            │       │           │ used the websearch tool    │                           │
│       │                            │       │           │ with the query "easy       │                           │
│       │                            │       │           │ chicken and broccoli       │                           │
│       │                            │       │           │ recipes"                   │                           │
│       │                            │       │           │ - The tool returned        │                           │
│       │                            │       │           │ relevant search results    │                           │
│       │                            │       │           │ with multiple recipe       │                           │
│       │                            │       │           │ sources and ideas          │                           │
│       │                            │       │           

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Evaluation Round 4: ToolSelectionAccuracyEvaluator

ToolSelectionAccuracyEvaluator validates that the agent selected the correct tools for the task.

In [22]:
# Create ToolSelectionAccuracyEvaluator
tool_selection_evaluator = ToolSelectionAccuracyEvaluator()

# Create dataset
tool_selection_dataset = Dataset[str, str](cases=test_cases, evaluator=tool_selection_evaluator)

# Run evaluation
tool_selection_report = tool_selection_dataset.run_evaluations(trace_task)


In [23]:
# Display ToolSelectionAccuracyEvaluator results
tool_selection_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ Cooking Technique Question         │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ Dietary Restriction Recipe         │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 1.00  │ ✅        │ The user has specifically  │ I have chicken and        │
│       │ Ingredients                │       │           │ stated they have chicken   │ broccoli. What can I      │
│       │                            │       │           │ and broccoli as            │ cook?                     │
│       │                            │       │           │ ingredients and asked      │                           │
│       │                            │       │           │ "What can I cook?" This is │                           │
│       │                            │       │           │ a clear request for recipe │                           │
│       │                            │       │           │ suggestions or cooking     │                           │
│       │                            │       │           │ ideas using these specific │                           │
│       │                            │       │           │ ingredients. The agent's   │                           │
│       │                            │       │           │ action to perform a        │                           │
│       │                            │       │           │ websearch with keywords    │                           │
│       │                            │       │           │ 'easy chicken and broccoli │                           │
│       │                            │       │           │ recipes' directly          │                           │
│       │                            │       │           │ addresses this request by: │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ 1. Targeting the exact     │                           │
│       │                            │       │           │ ingredients the user       │                           │
│       │                            │       │           │ mentioned (chicken and     │                           │
│       │                            │       │           │ broccoli)                  │                           │
│       │                            │       │           │ 2. Looking for recipes,    │                           │
│       │                            │       │           │ which is what the user is  │                           │
│       │                            │       │           │ implicitly asking for when │                           │
│       │                            │       │           │ they ask "what can I cook" │                           │
│       │                            │       │           │ 3. Including "easy" in the │                           │
│       │                            │       │           │ search terms, which is a   │                           │
│       │                            │       │           │ reasonable assumption for  │                           │
│       │                            │       │           │ someone looking for        │                           │
│       │                            │       │           │ cooking suggestions        │                           │
│       │                            │       │           │ 4. Using a web search tool │                           │
│       │                            │       │           │ to find current and varied │                           │
│       │                            │       │           

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Evaluation Round 5: ToolParameterAccuracyEvaluator

ToolParameterAccuracyEvaluator checks if the agent used tool parameters correctly (e.g., proper search keywords).

In [24]:
# Create ToolParameterAccuracyEvaluator
tool_parameter_evaluator = ToolParameterAccuracyEvaluator()

# Create dataset
tool_parameter_dataset = Dataset[str, str](cases=test_cases, evaluator=tool_parameter_evaluator)

# Run evaluation
tool_parameter_report = tool_parameter_dataset.run_evaluations(trace_task)


In [25]:
# Display ToolParameterAccuracyEvaluator results
tool_parameter_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 1   │ Cooking Technique Question         │ 1.00  │ ✅        │ ...    │ ...   │
├───────┼────────────────────────────────────┼───────┼───────────┼────────┼───────┤
│ ▶ 2   │ Dietary Restriction Recipe         │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 1.00  │ ✅        │ Analyzing the websearch    │ I have chicken and        │
│       │ Ingredients                │       │           │ tool-call parameter:       │ broccoli. What can I      │
│       │                            │       │           │                            │ cook?                     │
│       │                            │       │           │ 1. Parameter: 'keywords'   │                           │
│       │                            │       │           │ with value 'easy chicken   │                           │
│       │                            │       │           │ and broccoli recipes'      │                           │
│       │                            │       │           │                            │                           │
│       │                            │       │           │ Source analysis:           │                           │
│       │                            │       │           │ - The user explicitly      │                           │
│       │                            │       │           │ stated they have "chicken  │                           │
│       │                            │       │           │ and broccoli" as           │                           │
│       │                            │       │           │ ingredients                │                           │
│       │                            │       │           │ - The user asked "What can │                           │
│       │                            │       │           │ I cook?" which implies     │                           │
│       │                            │       │           │ they want recipes or       │                           │
│       │                            │       │           │ cooking suggestions        │                           │
│       │                            │       │           │ - The parameter value      │                           │
│       │                            │       │           │ directly incorporates both │                           │
│       │                            │       │           │ ingredients mentioned by   │                           │
│       │                            │       │           │ the user ("chicken and     │                           │
│       │                            │       │           │ broccoli")                 │                           │
│       │                            │       │           │ - The addition of "easy"   │                           │
│       │                            │       │           │ and "recipes" are          │                           │
│       │                            │       │           │ reasonable interpretations │                           │
│       │                            │       │           │ of what someone would      │                           │
│       │                            │       │           │ search for when asking     │                           │
│       │                            │       │           │ what they can cook with    │                           │
│       │                            │       │           │ specific ingredients       │                           │
│       │                            │       │           │ - No schema was provided   │                           │
│       │                            │       │           │ for the websearch tool,    │                           │
│       │                            │       │           

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Evaluation Round 6: TrajectoryEvaluator

TrajectoryEvaluator analyzes the sequence of actions (tool calls) the agent took to reach its conclusion.

In [26]:
# Create TrajectoryEvaluator with domain-specific rubric
trajectory_evaluator = TrajectoryEvaluator(
    rubric="""Agent should:
    1. Understand user's ingredients/dietary needs
    2. Search web with relevant recipe keywords
    3. Synthesize results into actionable recommendations
    Score: 1.0 if all steps present and logical, 0.5 if partially correct, 0.0 if flawed."""
)

# Create dataset (use single test case for demonstration)
trajectory_dataset = Dataset[str, str](cases=[test_cases[0]], evaluator=trajectory_evaluator)

# Task function with trajectory extraction
def trajectory_task(case: Case) -> dict:
    agent = Agent(
        system_prompt=RECIPE_BOT_SYSTEM_PROMPT,
        tools=[websearch],
        model=DEFAULT_MODEL
    )
    response = agent(case.input)

    # Update trajectory description
    trajectory_evaluator.update_trajectory_description(
        tools_use_extractor.extract_tools_description(agent)
    )

    # Extract trajectory from agent messages
    trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
    return {"output": str(response), "trajectory": trajectory}

# Run evaluation
trajectory_report = trajectory_dataset.run_evaluations(trajectory_task)


I'd be happy to help you find recipes that use chicken and broccoli! Let me search for some options for you.
Tool #1: websearch
Based on your search, I've found several delicious ways you can use your chicken and broccoli! Here are some recipe ideas:

## Chicken and Broccoli Recipe Options:

### 1. Chicken and Broccoli Stir-Fry
A quick and flavorful Asian-inspired dish where you can:
- Cook bite-sized chicken pieces until golden
- Add broccoli florets and stir-fry until crisp-tender
- Make a simple sauce with soy sauce, garlic, and ginger
- Serve over rice or noodles

### 2. Chicken and Broccoli Casserole
A comforting one-dish meal:
- Combine cooked chicken, steamed broccoli, and rice
- Add a creamy sauce made with cheese and milk
- Bake until bubbly and golden on top

### 3. Sheet Pan Honey Lime Chicken and Broccoli
An easy one-pan dinner:
- Marinate chicken in honey and lime
- Place chicken and broccoli on a sheet pan
- Bake together until chicken is cooked through and broccoli is ro

In [27]:
# Display TrajectoryEvaluator results
trajectory_report.run_display()

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                 Test Case Results                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ index ┃ name                               ┃ score ┃ test_pass ┃ reason ┃ input ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ ▶ 0   │ Recipe Search - Simple Ingredients │ 1.00  │ ✅        │ ...    │ ...   │
└───────┴────────────────────────────────────┴───────┴───────────┴────────┴───────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Test Case Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ index ┃ name                       ┃ score ┃ test_pass ┃ reason                     ┃ input                     ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ▼ 0   │ Recipe Search - Simple     │ 1.00  │ ✅        │ The agent perfectly        │ I have chicken and        │
│       │ Ingredients                │       │           │ executed all required      │ broccoli. What can I      │
│       │                            │       │           │ steps according to the     │ cook?                     │
│       │                            │       │           │ rubric: 1) Understood the  │                           │
│       │                            │       │           │ user's ingredients         │                           │
│       │                            │       │           │ (chicken and broccoli) and │                           │
│       │                            │       │           │ implicit need for cooking  │                           │
│       │                            │       │           │ suggestions, 2) Performed  │                           │
│       │                            │       │           │ a highly relevant web      │                           │
│       │                            │       │           │ search using the keywords  │                           │
│       │                            │       │           │ "easy chicken and broccoli │                           │
│       │                            │       │           │ recipes" which directly    │                           │
│       │                            │       │           │ matched the user's         │                           │
│       │                            │       │           │ request, and 3)            │                           │
│       │                            │       │           │ Effectively synthesized    │                           │
│       │                            │       │           │ the search results into 5  │                           │
│       │                            │       │           │ well-organized, actionable │                           │
│       │                            │       │           │ recipe recommendations     │                           │
│       │                            │       │           │ with clear cooking         │                           │
│       │                            │       │           │ instructions (stir-fry,    │                           │
│       │                            │       │           │ casserole, sheet pan,      │                           │
│       │                            │       │           │ teriyaki, and quinoa       │                           │
│       │                            │       │           │ variations). The output    │                           │
│       │                            │       │           │ was comprehensive,         │                           │
│       │                            │       │           │ helpful, and directly      │                           │
│       │                            │       │           │ addressed the user's       │                           │
│       │                            │       │           │ question with practical    │                           │
│       │                            │       │           │ cooking options.           │                           │
└───────┴────────────────────────────┴───────┴───────────┴────────────────────────────┴───────────────────────────┘

Enter the test case number to expand/collapse it, o to expand all, and c to collapse all (q to quit).:

### Summary: Comparing All Evaluators

Let's create a summary table comparing results from all six evaluators.

In [28]:
# Create summary comparison table
summary_data = {
    "Evaluator": [
        "OutputEvaluator",
        "HelpfulnessEvaluator",
        "GoalSuccessRateEvaluator",
        "ToolSelectionAccuracyEvaluator",
        "ToolParameterAccuracyEvaluator",
        "TrajectoryEvaluator"
    ],
    "Overall Score": [
        f"{output_report.overall_score:.2f}",
        f"{helpfulness_report.overall_score:.2f}",
        f"{goal_report.overall_score:.2f}",
        f"{tool_selection_report.overall_score:.2f}",
        f"{tool_parameter_report.overall_score:.2f}",
        f"{trajectory_report.overall_score:.2f}"
    ],
    "Type": [
        "Output-based",
        "Trace-based",
        "Trace-based",
        "Trace-based",
        "Trace-based",
        "Trajectory-based"
    ],
    "What It Measures": [
        "Response quality (ingredients, instructions, timing)",
        "User satisfaction (7-point scale)",
        "Goal completion (binary success/failure)",
        "Correct tool selection",
        "Correct tool parameters (keywords)",
        "Action sequence quality"
    ],
    "Requirements": [
        "None",
        "OpenTelemetry + Session",
        "OpenTelemetry + Session",
        "OpenTelemetry + Session",
        "OpenTelemetry + Session",
        "Trajectory extractor"
    ]
}

summary_df = pd.DataFrame(summary_data)
display(Markdown("### Built-In Evaluator Comparison"))
display(summary_df)

### Built-In Evaluator Comparison

,Evaluator,Overall Score,Type,What It Measures,Requirements
0,OutputEvaluator,0.37,Output-based,"Response quality (ingredients, instructions, t...",None
1,HelpfulnessEvaluator,0.83,Trace-based,User satisfaction (7-point scale),OpenTelemetry + Session
2,GoalSuccessRateEvaluator,1.00,Trace-based,Goal completion (binary success/failure),OpenTelemetry + Session
3,ToolSelectionAccuracyEvaluator,1.00,Trace-based,Correct tool selection,OpenTelemetry + Session
4,ToolParameterAccuracyEvaluator,1.00,Trace-based,Correct tool parameters (keywords),OpenTelemetry + Session
5,TrajectoryEvaluator,1.00,Trajectory-based,Action sequence quality,Trajectory extractor



#### Production Recommendations

For comprehensive agent evaluation:
1. Start with OutputEvaluator using domain-specific rubrics
2. Add HelpfulnessEvaluator and GoalSuccessRateEvaluator for user-centric metrics
3. Use tool evaluators if your agent has multiple tools or complex tool usage
4. Apply TrajectoryEvaluator for debugging and reasoning analysis

### Summary

You've successfully learned how to use built-in evaluators provided by Strands Evals.

In the next tutorial, you'll learn how to create custom evaluators for specialized evaluation needs beyond what the built-in evaluators provide.